In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scripts.download_data import download_coat, coat_is_available
from scripts.preprocess import load_coat, make_synthetic_mnar

RANDOM_SEED = 42
sns.set_theme(style="whitegrid", palette="muted")

## Notebook 00 — Data Exploration

**What this notebook does and why.**

Before fitting any model, we need to understand the data.
This notebook loads the Coat Shopping dataset (or falls back to a
synthetic MNAR surrogate) and visualises the key properties that
motivate Bayesian causal modelling:

1. **Rating-scale distribution** — do train vs test ratings differ? (MNAR signal)
2. **Sparsity** — how dense is the interaction matrix?
3. **Popularity bias** — are a few items responsible for most observations?

The Coat dataset has a biased *train* set (users self-selected what to rate)
and a uniform-random *test* set (each user was asked to rate a random subset).
This train/test split is precisely the MNAR structure we aim to correct.

In [ ]:
# ── Load data ────────────────────────────────────────────────────────
USE_COAT = False
try:
    download_coat("data/raw")
    coat = load_coat("data/raw")
    train_r = coat["train"]
    test_r  = coat["test"]
    train_mask = coat["train_mask"]
    test_mask  = coat["test_mask"]
    USE_COAT = True
    print(f"✓ Coat loaded — {train_r.shape[0]} users × {train_r.shape[1]} items")
except Exception as e:
    print(f"⚠️  Coat unavailable: {e}")
    print("Falling back to synthetic MNAR dataset.")
    syn = make_synthetic_mnar(
        n_users=500, n_items=200, n_factors=10, random_seed=RANDOM_SEED
    )
    train_r    = syn["train_ratings"]
    test_r     = syn["test_ratings"]
    train_mask = syn["train_mask"]
    test_mask  = syn["test_mask"]

n_users, n_items = train_r.shape
dataset_name = "Coat" if USE_COAT else "Synthetic MNAR"
print(f"Dataset   : {dataset_name}")
print(f"Shape     : {n_users} users × {n_items} items")
print(f"Train density: {train_mask.mean():.3f}  ({train_mask.sum():,} observations)")
print(f"Test  density: {test_mask.mean():.3f}  ({test_mask.sum():,} observations)")
print(f"Rating scale (train observed): {train_r[train_mask].min():.0f} – {train_r[train_mask].max():.0f}")

### 1 — Rating distributions: train vs test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for ax, ratings, mask, label, color in zip(
    axes,
    [train_r, test_r],
    [train_mask, test_mask],
    ["Train (biased)", "Test (unbiased)"],
    ["steelblue", "darkorange"],
):
    vals = ratings[mask]
    unique, counts = np.unique(vals, return_counts=True)
    ax.bar(unique, counts / counts.sum(), color=color, alpha=0.8, edgecolor="white")
    ax.set_title(label)
    ax.set_xlabel("Rating value")
    ax.set_ylabel("Fraction of observed ratings")
    ax.set_xticks(unique)

fig.suptitle(f"Rating distributions — {dataset_name}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/figures/00_rating_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

# Test: do distributions differ?
from scipy.stats import ks_2samp
ks_stat, ks_p = ks_2samp(train_r[train_mask], test_r[test_mask])
print(f"Kolmogorov–Smirnov test (train vs test): statistic={ks_stat:.3f}, p={ks_p:.4f}")
if ks_p < 0.05:
    print("→ Distributions differ significantly — MNAR bias confirmed.")
else:
    print("→ Distributions not significantly different.")

### 2 — Sparsity heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, mask, label in zip(
    axes,
    [train_mask, test_mask],
    ["Train mask", "Test mask"],
):
    # Subsample for visibility
    show_u = min(n_users, 100)
    show_i = min(n_items, 100)
    im = ax.imshow(
        mask[:show_u, :show_i].astype(float),
        aspect="auto", cmap="Blues", interpolation="nearest",
    )
    ax.set_title(f"{label}  (first {show_u}×{show_i})")
    ax.set_xlabel("Item index")
    ax.set_ylabel("User index")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(f"Sparsity heatmap — {dataset_name}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/figures/00_sparsity_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

### 3 — Item popularity distribution

In [ ]:
item_obs_counts = train_mask.sum(axis=0)  # observations per item in train

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(
    np.arange(n_items),
    np.sort(item_obs_counts)[::-1],
    color="steelblue", alpha=0.8, edgecolor="none",
)
axes[0].set_title("Item popularity (sorted, train)")
axes[0].set_xlabel("Item rank")
axes[0].set_ylabel("Number of observed ratings")

axes[1].hist(item_obs_counts, bins=30, color="steelblue", alpha=0.8, edgecolor="white")
axes[1].set_title("Popularity distribution histogram")
axes[1].set_xlabel("Observations per item (train)")
axes[1].set_ylabel("Number of items")

pct_top10 = item_obs_counts[np.argsort(item_obs_counts)[-int(n_items*0.1):]].sum() / item_obs_counts.sum()
print(f"Top 10% of items account for {pct_top10:.1%} of training observations → popularity bias present")

fig.suptitle(f"Item popularity — {dataset_name}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/figures/00_popularity_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### Summary

| Stat | Value |
|------|-------|
| Dataset | `{name}` |
| Users | `{n_users}` |
| Items | `{n_items}` |
| Train density | see output above |
| Test density  | see output above |

**Key takeaways:**
- The train and test rating distributions differ (MNAR bias) — this is why we need causal correction in Phase 2.
- Item popularity is highly skewed — a small fraction of items dominate training observations.
- The sparsity pattern confirms self-selection: users rate items they already like or know.

These observations motivate the full modelling pipeline in Phases 1–3.